In [1]:
# Bước 1: Xoá folder cũ nếu có
!rm -rf /kaggle/working/CS231_Video_Action_Recognition

# Bước 2: Clone (để yên, đừng bấm Ctrl+C, chờ nó chạy xong)
!git -c credential.helper='' clone --depth 1 \
    https://github.com/baotodale06/CS231_Video_Action_Recognition.git \
    /kaggle/working/CS231_Video_Action_Recognition

Cloning into '/kaggle/working/CS231_Video_Action_Recognition'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 21 (delta 1), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 105.03 MiB | 27.53 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [2]:
# Cài đặt các thư viện cần thiết
!pip install gradio decord torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 94.0 MB/s eta 0:00:00:00:01:01


In [3]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 93.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.2 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [4]:
import sys
import os
import torch
import torchvision.transforms as transforms
import gradio as gr
import numpy as np
import cv2  
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import urllib.request

# Download model file cho Pose Landmarker
model_path = "/tmp/pose_landmarker.task"
if not os.path.exists(model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
        model_path
    )
    print("Đã tải pose landmarker model.")

# Khởi tạo PoseLandmarker (thay thế mp.solutions.pose)
base_options = mp_python.BaseOptions(model_asset_path=model_path)
options = mp_vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,  # Xử lý từng frame
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
)
pose_landmarker = mp_vision.PoseLandmarker.create_from_options(options)
print("MediaPipe PoseLandmarker sẵn sàng!")

# ---------------------------------------------------------
# 1. IMPORT KIẾN TRÚC MODEL
# ---------------------------------------------------------
sys.path.append('/kaggle/working/CS231_Video_Action_Recognition')

from tsm_resnet50_model import TSM_Network 
from model import ViTGRU
from stgcn_mediapipe_model_2 import STGCN

# ---------------------------------------------------------
# 2. KHỞI TẠO 3 MODELS & LOAD TRỌNG SỐ (.pt)
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên thiết bị: {device}")

num_classes = 51   
num_segments = 16  

# Khởi tạo model
model_tsm = TSM_Network(num_classes=num_classes, n_segment=num_segments).to(device)
model_vitgru = ViTGRU(num_classes=num_classes).to(device)
model_stgcn = STGCN(num_class=num_classes, in_channels=3).to(device)

# Load trọng số (LƯU Ý: Bạn cần đổi đúng link cho ViT-GRU và ST-GCN)
path_tsm = "/kaggle/input/models/midzid/tsm-resnet50-best/pytorch/default/1/tsm_resnet50_best.pt"
path_vitgru = "/kaggle/input/datasets/midzid/vit-gru-best/vit_gru_best.pt" 
path_stgcn = "/kaggle/input/datasets/midzid/stgcn-mediapipe-best-2/stgcn_mediapipe_best_2.pt" 

model_tsm.load_state_dict(torch.load(path_tsm, map_location=device))
# Bỏ comment 2 dòng dưới khi bạn đã có file trọng số thực tế của 2 model này
model_vitgru.load_state_dict(torch.load(path_vitgru, map_location=device))
model_stgcn.load_state_dict(torch.load(path_stgcn, map_location=device))

model_tsm.eval()
model_vitgru.eval()
model_stgcn.eval()

# ---------------------------------------------------------
# 3. TỪ ĐIỂN NHÃN & TIỀN XỬ LÝ (PRE-PROCESSING)
# ---------------------------------------------------------
LABELS = [
    "brush_hair", "cartwheel", "catch", "chew", "clap", 
    "climb", "climb_stairs", "dive", "draw_sword", "dribble", 
    "drink", "eat", "fall_floor", "fencing", "flic_flac", 
    "golf", "handstand", "hit", "hug", "jump", 
    "kick", "kick_ball", "kiss", "laugh", "pick", 
    "pour", "pullup", "punch", "push", "pushup", 
    "ride_bike", "ride_horse", "run", "shake_hands", "shoot_ball", 
    "shoot_bow", "shoot_gun", "sit", "situp", "smile", 
    "smoke", "somersault", "stand", "swing_baseball", "sword", 
    "sword_exercise", "talk", "throw", "turn", "walk", 
    "wave"
]

# A. Tiền xử lý Ảnh RGB cho TSM và ViT-GRU
transform = transforms.Compose([
    transforms.ToPILImage(),             
    transforms.Resize((224, 224)),       
    transforms.ToTensor(),               
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

2026-05-18 14:31:48.745858: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779114708.979017      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779114709.056126      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779114709.580045      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779114709.580094      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779114709.580098      57 computation_placer.cc:177] computation placer alr

Đã tải pose landmarker model.
MediaPipe PoseLandmarker sẵn sàng!


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779114730.811664     157 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779114730.845629     157 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Đang chạy trên thiết bị: cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 186MB/s] 


In [5]:
# ✅ THÊM LẠI HÀM NÀY (bị thiếu trong code hiện tại)
def process_video_rgb(video_path, num_frames=16): 
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        return torch.zeros(1, num_frames, 3, 224, 224)
        
    frame_indices = np.linspace(0, max(total_frames - 1, 0), num_frames, dtype=int)
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(idx - 1, 0))
            ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(transform(frame_rgb))
        else:
            frames.append(frames[-1] if frames else torch.zeros(3, 224, 224))
    cap.release()
    
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else torch.zeros(3, 224, 224))
    
    video_tensor = torch.stack(frames[:num_frames]).unsqueeze(0)  # (1, T, C, H, W)
    return video_tensor
    


In [6]:
def process_video_skeleton(video_path, num_frames=64): 
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Xử lý trường hợp video lỗi/trống
    if total_frames == 0:
        cap.release()
        return torch.zeros(1, 3, num_frames, 33, 1)

    # ĐỊNH TUYẾN TRÍCH XUẤT
    if total_frames >= num_frames:
        # Nếu video dài: Lấy mẫu đều đặn (Uniform Sampling)
        frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    else:
        # Nếu video ngắn: Trích xuất toàn bộ số frame hiện có
        frame_indices = np.arange(total_frames)

    skeletons = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        
        # Fallback nếu đọc frame bị lỗi
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(idx - 1, 0))
            ret, frame = cap.read()

        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result = pose_landmarker.detect(mp_image)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]  
                # Trích xuất đúng x, y, visibility
                frame_skeleton = np.array(
                    [[lmk.x, lmk.y, lmk.visibility] for lmk in landmarks]
                )  
            else:
                frame_skeleton = np.zeros((33, 3))
        else:
            frame_skeleton = skeletons[-1] if skeletons else np.zeros((33, 3))

        skeletons.append(frame_skeleton)

    cap.release()
    
    # Chuyển list thành array có shape: (t_orig, 33, 3) hoặc (64, 33, 3)
    skeleton_arr = np.array(skeletons) 

    # XỬ LÝ TIMELOOP (LẶP VÒNG) CHO VIDEO NGẮN
    if total_frames < num_frames:
        t_orig = skeleton_arr.shape[0]
        if t_orig > 0:
            num_repeats = int(np.ceil(num_frames / t_orig))
            # Lặp vòng mảng tọa độ
            skeleton_tiled = np.tile(skeleton_arr, (num_repeats, 1, 1))
            # Cắt cho đúng đủ 64 frames
            skeleton_sampled = skeleton_tiled[:num_frames]
        else:
            skeleton_sampled = np.zeros((num_frames, 33, 3))
    else:
        skeleton_sampled = skeleton_arr

    # Chuyển vị và định hình Tensor: (T, V, C) -> (C, T, V)
    skeletons_transposed = np.transpose(skeleton_sampled, (2, 0, 1))       
    
    # Chuyển thành Pytorch Tensor và thêm Batch size + Số người: (1, 3, 64, 33, 1)
    tensor_skel = torch.tensor(skeletons_transposed, dtype=torch.float32)
    tensor_skel = tensor_skel.unsqueeze(0).unsqueeze(-1) 

    return tensor_skel

In [7]:
# =========================================================
# TÍNH NĂNG MỚI: HÀM TRÍCH XUẤT 16 FRAMES ĐỂ HIỂN THỊ UI
# =========================================================
def extract_frames_for_display(video_path, num_frames=16):
    if not video_path: 
        return []
        
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        return []

    frame_indices = np.linspace(0, max(total_frames - 1, 0), num_frames, dtype=int)
    frames_for_ui = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(idx - 1, 0))
            ret, frame = cap.read()
            
        if ret:
            # Chuyển BGR (OpenCV) sang RGB để Gradio hiển thị đúng màu
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames_for_ui.append(frame_rgb)
            
    cap.release()
    
    # Padding nếu video quá ngắn không đủ lấy
    while len(frames_for_ui) < num_frames and len(frames_for_ui) > 0:
        frames_for_ui.append(frames_for_ui[-1])
        
    return frames_for_ui[:num_frames]

In [8]:
# ---------------------------------------------------------
# 4. CÁC HÀM DỰ ĐOÁN ĐỘC LẬP
# ---------------------------------------------------------
def get_prediction_text(outputs):
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    predicted_idx = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_idx].item()
    return f"Hành động: **{LABELS[predicted_idx]}**\nĐộ chính xác: {confidence*100:.2f}%"

def predict_tsm(video_filepath):
    if not video_filepath: return "Vui lòng tải video."
    try:
        input_tensor = process_video_rgb(video_filepath).to(device)
        with torch.no_grad():
            return get_prediction_text(model_tsm(input_tensor))
    except Exception as e: return f"Lỗi: {str(e)}"

def predict_vitgru(video_filepath):
    if not video_filepath: return "Vui lòng tải video."
    try:
        input_tensor = process_video_rgb(video_filepath).to(device)
        with torch.no_grad():
            return get_prediction_text(model_vitgru(input_tensor))
    except Exception as e: return f"Lỗi: {str(e)}"

def predict_stgcn(video_filepath):
    if not video_filepath: return "Vui lòng tải video."
    try:
        input_tensor = process_video_skeleton(video_filepath).to(device)
        with torch.no_grad():
            return get_prediction_text(model_stgcn(input_tensor))
    except Exception as e: return f"Lỗi: {str(e)}"



In [9]:
# ---------------------------------------------------------
# 5. XÂY DỰNG GIAO DIỆN GRADIO BLOCKS
# ---------------------------------------------------------
gr.close_all() 

with gr.Blocks() as demo:
    gr.Markdown("<center><h1>🚀 Hệ Thống Nhận Diện Hành Động (Đa Mô Hình)</h1></center>")
    gr.Markdown("Tải video lên và trải nghiệm sức mạnh của 3 cấu trúc mạng khác nhau: CNN (TSM), Transformer + RNN (ViT-GRU), và Graph Neural Network (ST-GCN).")
    
    # --- TAB 1: TSM ---
    with gr.Tab("🎞️ TSM (ResNet50)"):
        with gr.Row():
            with gr.Column():
                vid_tsm = gr.Video(label="Upload Video Test")
                # THÊM NÚT PREPROCESS
                btn_prep_tsm = gr.Button("🔍 Preprocess (Xem 16 Frames)", variant="secondary")
                btn_tsm = gr.Button("Dự đoán bằng TSM", variant="primary")
            with gr.Column():
                out_tsm = gr.Markdown(label="Kết quả")
                # THÊM GALLERY ĐỂ HIỂN THỊ FRAMES (hiển thị dạng lưới 4x4)
                gal_tsm = gr.Gallery(label="16 Frames đầu vào", columns=4, rows=4, height="auto")
                
        btn_prep_tsm.click(fn=extract_frames_for_display, inputs=vid_tsm, outputs=gal_tsm)
        btn_tsm.click(fn=predict_tsm, inputs=vid_tsm, outputs=out_tsm)

    # --- TAB 2: ViT-GRU ---
    with gr.Tab("👁️ ViT-GRU"):
        with gr.Row():
            with gr.Column():
                vid_vit = gr.Video(label="Upload Video Test")
                # THÊM NÚT PREPROCESS
                btn_prep_vit = gr.Button("🔍 Preprocess (Xem 16 Frames)", variant="secondary")
                btn_vit = gr.Button("Dự đoán bằng ViT-GRU", variant="primary")
            with gr.Column():
                out_vit = gr.Markdown(label="Kết quả")
                # THÊM GALLERY ĐỂ HIỂN THỊ FRAMES
                gal_vit = gr.Gallery(label="16 Frames đầu vào", columns=4, rows=4, height="auto")
                
        btn_prep_vit.click(fn=extract_frames_for_display, inputs=vid_vit, outputs=gal_vit)
        btn_vit.click(fn=predict_vitgru, inputs=vid_vit, outputs=out_vit)

    # --- TAB 3: ST-GCN ---
    with gr.Tab("🦴 ST-GCN (Skeleton)"):
        with gr.Row():
            with gr.Column():
                vid_gcn = gr.Video(label="Upload Video Test")
                # THÊM NÚT PREPROCESS (Model này dùng 64 frames, nhưng ta vẫn chiếu 16 frames đại diện cho gọn UI)
                btn_prep_gcn = gr.Button("🔍 Preprocess (Xem 16 Frames)", variant="secondary")
                btn_gcn = gr.Button("Dự đoán bằng ST-GCN", variant="primary")
            with gr.Column():
                out_gcn = gr.Markdown(label="Kết quả")
                # THÊM GALLERY ĐỂ HIỂN THỊ FRAMES
                gal_gcn = gr.Gallery(label="16 Frames trích xuất", columns=4, rows=4, height="auto")
                
        btn_prep_gcn.click(fn=extract_frames_for_display, inputs=vid_gcn, outputs=gal_gcn)
        btn_gcn.click(fn=predict_stgcn, inputs=vid_gcn, outputs=out_gcn)

In [10]:
# ---------------------------------------------------------
# 6. KHỞI CHẠY APP
# ---------------------------------------------------------
demo.launch(server_name="0.0.0.0", share=True)

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://f39b6c1688826354d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
!pip show mediapipe  # Kiểm tra version hiện tại
!pip install mediapipe==0.10.9 --quiet

Name: mediapipe
Version: 0.10.35
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: absl-py, certifi, flatbuffers, matplotlib, numpy, opencv-contrib-python, sounddevice
Required-by: 
ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.9 (from versions: 0.10.13, 0.10.14, 0.10.15, 0.10.18, 0.10.20, 0.10.21, 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)
ERROR: No matching distribution found for mediapipe==0.10.9
